In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import ElementNotInteractableException

from pathlib import Path
import time
import pandas as pd

browser = webdriver.Chrome()

# Asking to input starting and ending year for our scrape, in my analysis the starting year was 1990, ending year was 2025
starting_year = input("Input starting year (1917-2025)")
ending_year = input("Input ending year(1917-2025)")

# Creating list with years we want the data for, based on the input years.
scrape_years_list= list(range(int(starting_year), int(ending_year) + 1))

# Creating folder to put the scraped files into
folder_path = Path.cwd()/"NHL Season CSVs"
folder_path.mkdir(parents=True, exist_ok=True)

# Creating loop to go through list of the years we want to scrape and create .csv file with the data table for each year
for i in scrape_years_list:

    # Loading URL of the correct season
    browser.get(f"https://www.hockey-reference.com/leagues/NHL_{i}.html")
    browser.maximize_window()

    # Clicking Accept Cookies if present, otherwise pass
    try:
        browser.find_element(By.XPATH, '/html/body/div[2]/div[2]/div[2]/button[1]').click()
    except NoSuchElementException as exception:
        pass
    except ElementNotInteractableException as exception:
        pass

    # Safety wait for the page to fully load, possible subscription popup to appear and to comply with scraping request limits (https://www.sports-reference.com/bot-traffic.html)
    time.sleep(10)

    # Closing the subscription popup if present, otherwise pass
    try: 
        browser.find_element(By.XPATH, '//*[@id="modal-close"]').click()
    except NoSuchElementException as exception:
        pass
    except ElementNotInteractableException as exception:
        pass

    # Finding element that needs to be hovered over
    try:
        export_dropdown = browser.find_element(By.XPATH, '/html/body/div[4]/div[6]/div[5]/div[1]/div/ul/li[1]/span')
    # Element can't be found in some seasons (2004-05 lockout for example), continue with the loop if so.
    except NoSuchElementException as exception:
        continue

    # Finding element that needs to be clicked
    click_csv = browser.find_element(By.XPATH, '/html/body/div[4]/div[6]/div[5]/div[1]/div/ul/li[1]/div/ul/li[3]/button')

    # Action chain that hovers the dropdown menu and clicks button that converts the table to csv
    ActionChains(browser).move_to_element(export_dropdown).click(click_csv).perform()

    # Finding element with the season table 
    table = browser.find_element(By.ID, 'csv_stats').text

    # Removing first five lines of the table that don't include any useful data
    cleaned_table = "\n".join(table.splitlines()[5:])

    # Saving the table 
    (folder_path/f"NHL_season_{i}.csv").write_text(cleaned_table)
    
    # Loading the CSV to Pandas for further cleaning
    df = pd.read_csv(folder_path/f"NHL_season_{i}.csv")

    # Fixing the 2nd Column name that's currently missing
    df = df.rename(columns={df.columns[1]: 'TEAM'})

    #Changing datatype of the Rk column from float to int
    df["Rk"] = df["Rk"].astype("Int64")

    # Adding a column with the season year
    df.insert(0, "YEAR", f'{i}')

    # Teams that made play-off are marked with asterisk next to their name, I want to create special column for it and drop the asterisk to standardize the team names
    # Creating column "PO" (play-off) and setting default value to "NO"
    df["PO"] = "NO"

    # Changing the 2nd column datatype from object to string
    df["TEAM"] = df["TEAM"].astype("string")

    # changing "PO" value to "YES" if team name includes asterisk
    df.loc[df["TEAM"].str.contains('\*'), "PO"] = "YES"

    # Droping the asterisk from team name
    df["TEAM"] = df["TEAM"].str.replace('*', '')

    # Saving cleaned file
    df.to_csv(folder_path/f"NHL_season_{i}.csv", index=False)
browser.quit()